In [3]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [5]:
data = [
    (101, "2025-06-01", "West", "Electronics", 2500, "Active", "Jaipur", 25, "Premium", "2025-06-21 10:15:00", "john", "john@gmail.com", "S101"),
    (101, "2025-06-01", "West", "Electronics", 2500, None, "Jaipur", 25, "Premium", "2025-06-21 10:15:00", "john", "john@gmail.com", "S101"),
    (102, "2025-06-02", "West", "Furniture", None, "Pending", "Delhi", 29, "Premium", "2025-06-21 11:30:00", "", "alice@gmail.com", "S102"),
    (103, "2025-06-03", "East", "Electronics", 3000, None, "Mumbai", 35, "Basic", "2025-06-21 12:00:00", "mike", None, "S103"),
    (104, "2025-06-04", "West", "Clothing", 1500, "Active", "Delhi", 22, "Premium", "2025-06-21 13:00:00", "rohit", "rohit@gmail.com", "S102")
]

columns = [
    "user_id",
    "transaction_date",
    "region",
    "product_category",
    "price",
    "status",
    "city",
    "age",
    "subscription",
    "raw_timestamp",
    "username",
    "email",
    "store_id"
]

df = spark.createDataFrame(data, columns)

df.show(truncate=False)

+-------+----------------+------+----------------+-----+-------+------+---+------------+-------------------+--------+---------------+--------+
|user_id|transaction_date|region|product_category|price|status |city  |age|subscription|raw_timestamp      |username|email          |store_id|
+-------+----------------+------+----------------+-----+-------+------+---+------------+-------------------+--------+---------------+--------+
|101    |2025-06-01      |West  |Electronics     |2500 |Active |Jaipur|25 |Premium     |2025-06-21 10:15:00|john    |john@gmail.com |S101    |
|101    |2025-06-01      |West  |Electronics     |2500 |NULL   |Jaipur|25 |Premium     |2025-06-21 10:15:00|john    |john@gmail.com |S101    |
|102    |2025-06-02      |West  |Furniture       |NULL |Pending|Delhi |29 |Premium     |2025-06-21 11:30:00|        |alice@gmail.com|S102    |
|103    |2025-06-03      |East  |Electronics     |3000 |NULL   |Mumbai|35 |Basic       |2025-06-21 12:00:00|mike    |NULL           |S103    |

Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [6]:
df_clean = df.dropDuplicates(
    ["user_id", "transaction_date"]
)

df_clean.show()

+-------+----------------+------+----------------+-----+-------+------+---+------------+-------------------+--------+---------------+--------+
|user_id|transaction_date|region|product_category|price| status|  city|age|subscription|      raw_timestamp|username|          email|store_id|
+-------+----------------+------+----------------+-----+-------+------+---+------------+-------------------+--------+---------------+--------+
|    101|      2025-06-01|  West|     Electronics| 2500| Active|Jaipur| 25|     Premium|2025-06-21 10:15:00|    john| john@gmail.com|    S101|
|    102|      2025-06-02|  West|       Furniture| NULL|Pending| Delhi| 29|     Premium|2025-06-21 11:30:00|        |alice@gmail.com|    S102|
|    103|      2025-06-03|  East|     Electronics| 3000|   NULL|Mumbai| 35|       Basic|2025-06-21 12:00:00|    mike|           NULL|    S103|
|    104|      2025-06-04|  West|        Clothing| 1500| Active| Delhi| 22|     Premium|2025-06-21 13:00:00|   rohit|rohit@gmail.com|    S102|

 Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [7]:
# price column is used as sale_amount in the sample dataset
df_q4 = (
    df.filter(col("region") == "West")
      .groupBy("product_category")
      .agg(avg("price").alias("avg_sale_amount"))
)

df_q4.show()

+----------------+---------------+
|product_category|avg_sale_amount|
+----------------+---------------+
|     Electronics|         2500.0|
|        Clothing|         1500.0|
|       Furniture|           NULL|
+----------------+---------------+



Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

In [8]:
df_q5 = df.na.fill({
    "status": "Unknown"
})

df_q5.select(
    "user_id",
    "status"
).show()

+-------+-------+
|user_id| status|
+-------+-------+
|    101| Active|
|    101|Unknown|
|    102|Pending|
|    103|Unknown|
|    104| Active|
+-------+-------+



Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [9]:
df_q6 = (
    df.groupBy("city")
      .agg(count("*").alias("city_count"))
      .filter(col("city_count") > 1)
)

df_q6.show()
## NOTE:# Using >1 because the sample dataset is small.
# In a real dataset, the condition would be >100.

+------+----------+
|  city|city_count|
+------+----------+
|Jaipur|         2|
| Delhi|         2|
+------+----------+



Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [10]:
df_q8 = df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

df_q8.show()

+-------+----------------+------+----------------+-----+-------+------+---+------------+-------------------+--------+---------------+--------+
|user_id|transaction_date|region|product_category|price| status|  city|age|subscription|      raw_timestamp|username|          email|store_id|
+-------+----------------+------+----------------+-----+-------+------+---+------------+-------------------+--------+---------------+--------+
|    101|      2025-06-01|  West|     Electronics| 2500| Active|Jaipur| 25|     Premium|2025-06-21 10:15:00|    john| john@gmail.com|    S101|
|    101|      2025-06-01|  West|     Electronics| 2500|   NULL|Jaipur| 25|     Premium|2025-06-21 10:15:00|    john| john@gmail.com|    S101|
|    102|      2025-06-02|  West|       Furniture| NULL|Pending| Delhi| 29|     Premium|2025-06-21 11:30:00|        |alice@gmail.com|    S102|
|    104|      2025-06-04|  West|        Clothing| 1500| Active| Delhi| 22|     Premium|2025-06-21 13:00:00|   rohit|rohit@gmail.com|    S102|

Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [11]:
df_q10 = (
    df.withColumn(
        "event_time",
        col("raw_timestamp").cast(TimestampType())
    )
    .drop("raw_timestamp")
)

df_q10.show()

+-------+----------------+------+----------------+-----+-------+------+---+------------+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|price| status|  city|age|subscription|username|          email|store_id|         event_time|
+-------+----------------+------+----------------+-----+-------+------+---+------------+--------+---------------+--------+-------------------+
|    101|      2025-06-01|  West|     Electronics| 2500| Active|Jaipur| 25|     Premium|    john| john@gmail.com|    S101|2025-06-21 10:15:00|
|    101|      2025-06-01|  West|     Electronics| 2500|   NULL|Jaipur| 25|     Premium|    john| john@gmail.com|    S101|2025-06-21 10:15:00|
|    102|      2025-06-02|  West|       Furniture| NULL|Pending| Delhi| 29|     Premium|        |alice@gmail.com|    S102|2025-06-21 11:30:00|
|    103|      2025-06-03|  East|     Electronics| 3000|   NULL|Mumbai| 35|       Basic|    mike|           NULL|    S103|2025-06-21 12:00:00|

Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [12]:
df_q12 = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

df_q12.show()

+-------+----------------+------+----------------+-----+------+------+---+------------+-------------------+--------+---------------+--------+
|user_id|transaction_date|region|product_category|price|status|  city|age|subscription|      raw_timestamp|username|          email|store_id|
+-------+----------------+------+----------------+-----+------+------+---+------------+-------------------+--------+---------------+--------+
|    101|      2025-06-01|  West|     Electronics| 2500|Active|Jaipur| 25|     Premium|2025-06-21 10:15:00|    john| john@gmail.com|    S101|
|    101|      2025-06-01|  West|     Electronics| 2500|  NULL|Jaipur| 25|     Premium|2025-06-21 10:15:00|    john| john@gmail.com|    S101|
|    104|      2025-06-04|  West|        Clothing| 1500|Active| Delhi| 22|     Premium|2025-06-21 13:00:00|   rohit|rohit@gmail.com|    S102|
+-------+----------------+------+----------------+-----+------+------+---+------------+-------------------+--------+---------------+--------+



Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [13]:
df.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    mean("price").alias("avg_price")
).show()

+---------+---------+---------+
|min_price|max_price|avg_price|
+---------+---------+---------+
|     1500|     3000|   2375.0|
+---------+---------+---------+



Q15: Write a final processing pipeline that filters out duplicates, fills null prices with 0, and groups by store_id to calculate total revenue.

In [15]:
df_q15 = (
    df.dropDuplicates()
      .na.fill({"price": 0})
      .groupBy("store_id")
      .agg(sum("price").alias("total_revenue"))
)

df_q15.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|    S102|         1500|
|    S101|         5000|
|    S103|         3000|
+--------+-------------+



Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Answer:

Traditional MapReduce has several limitations that make it less suitable for modern big data applications:

It writes intermediate results to disk after every processing stage, resulting in high disk I/O.
It is slow for iterative tasks such as machine learning and graph processing.
It has high latency and is not suitable for real-time analytics.
Multiple MapReduce jobs are often required for complex workflows.
Programming and maintenance are more complex.

Apache Spark overcomes these limitations by using in-memory processing, providing faster execution, lower latency, and support for advanced analytics, machine learning, and streaming.

Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Answer:

Spark uses in-memory computing by storing intermediate datasets in RAM instead of writing them to disk after every operation. Machine learning algorithms often require multiple iterations over the same data. Since Spark keeps data in memory, it avoids repeated disk reads and writes, significantly reducing execution time and improving performance compared to traditional disk-based systems such as MapReduce.

Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

Answer:

Spark DataFrames are immutable, which means they cannot be modified after they are created. Any operation such as dropping columns, renaming columns, filtering rows, or handling null values creates a new DataFrame rather than changing the original one. This ensures data consistency and fault tolerance while processing large datasets.

Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

Answer:

Handling null values before performing aggregations is important because null values can lead to inaccurate or misleading results. Missing values may affect calculations such as sum(), average(), minimum(), and maximum(). Replacing or removing null values before aggregation helps ensure that the computed statistics accurately represent the dataset.

Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

Answer:

Shuffle is the process of redistributing data across partitions so that records with the same key are brought together. Operations such as groupBy(), join(), and distinct() require data to move between partitions across the cluster.

Shuffle is considered a wide transformation because data from multiple partitions must be exchanged over the network. This movement of data increases processing time and resource usage, making shuffle one of the most expensive operations in Spark.

Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

Answer:

Using inferSchema=true on datasets with inconsistent or messy date formats can cause Spark to incorrectly identify the data type of a column. Some date values may be interpreted as strings, while others may become null due to parsing errors. This can lead to data quality issues, incorrect analysis, and unexpected results. Therefore, it is often safer to load dates as strings first, clean them, and then explicitly cast them to the appropriate date or timestamp type.
